# Line versus Word Qwen performance

In [5]:
!pip install pytesseract

## Doctr and line creation algorithm

In [2]:
# see doctr_line_algorithm notebook for the original code
# This script is a modified version of the original

import cv2 as cv
import numpy as np
from doctr.models import detection_predictor
import matplotlib.pyplot as plt

image = cv.imread("/home/bas/Documents/Visual Code Repo's/BelHisFirm-BelHisHAAI/experiments/test/test-Yves_026_None.png", cv.IMREAD_COLOR)

def warp_function(image):
    gray = image if image.ndim == 2 else cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    edges = cv.Canny(gray, 50, 150, apertureSize=3)
    lines = cv.HoughLinesP(edges, 1, np.pi/180, threshold=100,
                           minLineLength=100, maxLineGap=10)

    angles = []
    if lines is not None:
        for x1, y1, x2, y2 in lines[:, 0, :]:
            angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
            angles.append(angle)

    if not angles:
        return image

    median_angle = float(np.median(angles))

    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv.getRotationMatrix2D(center, median_angle, 1.0).astype(np.float32)

    rotated = cv.warpAffine(
        image, M, (w, h),
        flags=cv.INTER_CUBIC,
        borderMode=cv.BORDER_REPLICATE
    )
    return rotated

image = warp_function(image)

model = detection_predictor('db_resnet50', pretrained=True, assume_straight_pages=False, preserve_aspect_ratio=True)

out = model([image])
words_array = out[0]['words']


/home/bas/Documents/Visual Code Repo's/BelHisFirm-BelHisHAAI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class BoxtoLine:

    def __init__(self):
        self.bbox_list = []
        self.line_bbox_list = []

    @staticmethod
    def clamp_box(x0, y0, x1, y1):
            x0, x1 = sorted([x0, x1])
            y0, y1 = sorted([y0, y1])
            return x0, y0, x1, y1
    
    @staticmethod
    def vert_iou_calc(bbox1, bbox2):
        x1, y1, x2, y2 = bbox1
        x3, y3, x4, y4 = bbox2

        if x1 <= x3 and x2 >= x4 and y1 <= y3 and y2 >= y4:
            return 1.0
        else:
            inter_w = max(0.0, 0.1)
            inter_h = max(0.0, min(y2, y4) - max(y1, y3))

            inter = inter_w * inter_h

            area_a = max(0.0, (0.1) * max(0.0, (y2 - y1)))
            area_b = max(0.0, (0.1) * max(0.0, (y4 - y3)))

            union = area_a + area_b - inter

            return inter / union 
    
    @staticmethod
    def resolve_bbox(bbox1, bbox2):

        x1, y1, x2, y2 = bbox1
        x3, y3, x4, y4 = bbox2
        bbox3 = (min(x1, x3), min(y1, y3), max(x2, x4), max(y2, y4))

        return bbox3

    def transform_boxes_to_line(self, words_array): 
        # restructure
        self.bbox_list = [(box[0][0], box[0][1], box[2][0], box[2][1]) for box in words_array]
        # purge small boxes
        self.bbox_list = [box for box in self.bbox_list if (box[2] - box[0]) > 0.01 or (box[3] - box[1]) > 0.01]
        # clamp boxes
        self.bbox_list = [self.clamp_box(x0, y0, x1, y1) for (x0, y0, x1, y1) in self.bbox_list]
        # Sort by vertical position and then horizontal position (halfway point)
        self.bbox_list.sort(key=lambda b: (0.5*(b[1]+b[3]), b[0])) 

        word_counter = 1

        for idx, bbox in enumerate(self.bbox_list):
            if idx > 0:
                iou = self.vert_iou_calc(line, bbox)
                if iou > 0.5: 
                    line = self.resolve_bbox(line, bbox)
                    word_counter += 1
                else:
                    line_token = (line, word_counter)
                    self.line_bbox_list.append(line_token)
                    line = bbox
                    word_counter = 1
            else:
                line = bbox
            if idx == len(self.bbox_list) - 1:
                self.line_bbox_list.append((line_token))


# Create an instance of the class and transform the boxes
box_to_line = BoxtoLine()
box_to_line.transform_boxes_to_line(words_array)

In [ ]:
import pytesseract

def extract_line_pytesseract(line_bbox_list, box_list, image):

    h, w = image.shape[:2]
    lines_of_texts = []
    line_tess = None

    current_counted = 0

    total_word_count = sum(line[1] for line in line_bbox_list)

    for line in line_bbox_list:
        word_counter = line[1]
        expected_counted = word_counter + current_counted

        while current_counted <= total_word_count:
            try:
                x1, y1, x2, y2 = box_list[current_counted]
                x1 = int(x1 * w)
                x2 = int(x2 * w)
                y1 = int(y1 * h)
                y2 = int(y2 * h)

                crop = image[y1:y2, x1:x2]
                text = pytesseract.image_to_string(crop, lang='fra', config='--psm 13')

                if line_tess is None:
                    line_tess = text
                else:
                    line_tess = line_tess + " " + text
                if current_counted == expected_counted:
                    lines_of_texts.append(line_tess)
                    line_tess = None
                current_counted += 1
                print(text)
            except:
                pass
            

    return lines_of_texts

lines_of_texts = extract_line_pytesseract(box_to_line.line_bbox_list, box_to_line.bbox_list, image)

print(lines_of_texts)

    


SOCTÉTÉE.

"AUX

DOCUMENTS

RELATIFS

ACTES

‘ET

SPÉCIAL

DES

RECUKIL

382

Mnounement

du

portefeuille

ensemble

effets,

2.399

755.677

159

fr.

4882,

décembre

51

au

Il existait

en

portefeuille,

34,905

20,471,285

97

“entré

1883

en

u

‘est

91.996.960

PTS

57,502

90 640 95

355,262

26

1883.

sorti

un

est

en

386 _009 c

2,040

fe.

décembre

4885

54

au

En

portefeuille

“de

et

‘Compte

‘pertes.

profits

91.763 89

“fr.

de

4883,

«est

31

décembre

de

créditeur

ce

compte

au

Le

‘solde

déduire

A

fr

eo 740

05

‘généraux

Frais

8.269

48

9%

55.509

portefeuille.

9°

du

Réescompte

4,300

Fonds

ete.)

de

5°

prévision

(patente,

Pénéfices

56,261 59

nets.

OR

2,815

8.

Ala

réserve

statutaire

D.

“Ce

‘fr.

53. LAB

381

“Reste

conformément

à

Particle

59

dés

“statuts

suit,

A

comme

répartir

34 300

‘fr.

francs

de

857,500

capital

versé

(soit

y,

11

©.)

68

actionnaires

sur

‘aux

et.

Pp.

40

44 115

28

eaà

la
